# Phase 0 — Lesson 06: Python Environments
> Dependency hell is real. Virtual environments are the cure.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 06
**Type:** Build | **Languages:** Shell | **Time:** ~30 minutes
**Prerequisites:** Phase 0, Lesson 01

This lesson explains *why* dependency conflicts happen, *how* environments isolate them, and *which* tool to use when. Run the code cells to see visualizations.


# The Problem: Dependency Hell

---


You install PyTorch 2.4 for a fine-tuning project. Next week, a different project needs PyTorch 2.1 because its CUDA build is pinned. You upgrade globally, and the first project breaks. You downgrade, and the second one breaks.

This is **dependency hell**. It happens constantly in AI/ML work because:

- PyTorch, JAX, and TensorFlow each ship their own CUDA bindings
- Model libraries pin specific framework versions
- A global `pip install` overwrites whatever was there before
- CUDA 11.8 builds don't work with CUDA 12.x drivers (and vice versa)

The fix: every project gets its own isolated environment with its own packages.


# Visual: Dependency Hell — One Global Python

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

# System Python box
rect = patches.Rectangle((0.1, 0.4), 0.8, 0.4, facecolor='#e74c3c', edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(0.5, 0.6, 'System Python (global)', ha='center', va='center', color='white', fontweight='bold', fontsize=12)

# Conflicting packages inside
ax.text(0.2, 0.5, 'torch 2.4', ha='center', fontsize=10, color='white', fontweight='bold')
ax.text(0.5, 0.5, 'torch 2.1', ha='center', fontsize=10, color='white', fontweight='bold')
ax.text(0.8, 0.5, '???', ha='center', fontsize=10, color='white', fontweight='bold')

# Conflict arrow
ax.annotate('', xy=(0.5, 0.32), xytext=(0.5, 0.42),
            arrowprops=dict(arrowstyle='->', color='black', lw=3))
ax.text(0.5, 0.25, 'CONFLICT: only one torch version can exist globally', ha='center',
        fontsize=11, fontweight='bold', color='#e74c3c')

# Projects
ax.text(0.2, 0.75, 'Project A needs', ha='center', fontsize=9)
ax.text(0.8, 0.75, 'Project B needs', ha='center', fontsize=9)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Without Virtual Environments: One Global Python, Conflicting Needs", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **The root cause:** The system Python has only one `site-packages` folder. Two projects needing different versions of the same package cannot coexist.


# The Concept: Virtual Environments Isolate Projects

---


A virtual environment is a folder containing its own Python interpreter and its own `site-packages`. Each project gets its own folder. Nothing leaks between them.

| Without Environments | With Environments |
|----------------------|-------------------|
| One global Python | Each project has its own `.venv/` |
| One version per package | Each project pins its own versions |
| Installing breaks other projects | Projects are independent |
| `pip install` affects everything | `pip install` affects only this project |


# Visual: With Virtual Environments — Isolation

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Project A
rect_a = patches.Rectangle((0.05, 0.35), 0.4, 0.5, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect_a)
ax.text(0.25, 0.8, 'Project A (.venv/)', ha='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.15, 0.65, 'torch 2.4.0', ha='center', fontsize=9, color='white')
ax.text(0.15, 0.58, 'CUDA 12.4', ha='center', fontsize=9, color='white')
ax.text(0.15, 0.51, 'transformers 4.44', ha='center', fontsize=9, color='white')
ax.text(0.15, 0.44, 'numpy 1.26', ha='center', fontsize=9, color='white')

# Project B
rect_b = patches.Rectangle((0.55, 0.35), 0.4, 0.5, facecolor='#2ecc71', edgecolor='black', linewidth=2)
ax.add_patch(rect_b)
ax.text(0.75, 0.8, 'Project B (.venv/)', ha='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.65, 0.65, 'torch 2.1.0', ha='center', fontsize=9, color='white')
ax.text(0.65, 0.58, 'CUDA 11.8', ha='center', fontsize=9, color='white')
ax.text(0.65, 0.51, 'diffusers 0.28', ha='center', fontsize=9, color='white')
ax.text(0.65, 0.44, 'numpy 1.24', ha='center', fontsize=9, color='white')

# Separation line
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=2)
ax.text(0.5, 0.15, 'ISOLATED: neither project sees the other's packages', ha='center',
        fontsize=11, fontweight='bold', color='#2c3e50')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("With Virtual Environments: Each Project Is Independent", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Key insight:** Project A and Project B can use different PyTorch, different CUDA, even different NumPy versions. They never see each other.


# Three Tools for Creating Environments

---


| Tool | Best For | Speed | Non-Python Deps |
|------|----------|-------|-----------------|
| **uv** (recommended) | General AI/ML projects | Fastest (10-100x pip) | No |
| **venv** (built-in) | When you can't install uv | Slow | No |
| **conda** | CUDA toolkits, C libraries, shared clusters | Medium | Yes |

For this course, **uv** is recommended. It handles Python versions, virtual environments, and dependency resolution in one tool.


# Option 1: uv venv (Recommended)

---


```bash
# Install uv (one-time)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Install Python 3.12 via uv
uv python install 3.12

# Create and activate a virtual environment
cd your-project
uv venv
source .venv/bin/activate

# Install packages
uv pip install torch numpy

# Or initialize a project with pyproject.toml
uv init my-ai-project
cd my-ai-project
uv add torch numpy matplotlib
```

**Why uv:** It's written in Rust, resolves dependencies in parallel, and replaces pip, venv, and pyenv in one tool.


# Option 2: venv (Built-in)

---


If you can't install `uv`, Python ships with `venv`:

```bash
python3 -m venv .venv
source .venv/bin/activate     # Linux/macOS
.venv\\Scripts\\activate        # Windows

pip install torch numpy
```

Slower than `uv`, but works everywhere Python is installed — no extra installation needed.


# Option 3: conda (When You Need Non-Python Deps)

---


Conda manages non-Python dependencies like CUDA toolkits, cuDNN, and C libraries. Use it when:

- You need a specific CUDA toolkit version without installing it system-wide
- You're on a shared cluster where you can't install system packages
- A library's install instructions say "use conda"

```bash
# Install miniconda (not the full Anaconda)
curl -LsSf https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o miniconda.sh
bash miniconda.sh -b

conda create -n myproject python=3.12
conda activate myproject

conda install pytorch torchvision torchaudio pytorch-cuda=12.4 -c pytorch -c nvidia
```

**One rule:** If you use conda for an environment, use conda for **all** packages in that environment. Mixing `pip install` into a conda env causes dependency conflicts that are painful to debug.


# Visual: Choosing the Right Tool

---


In [ ]:
import matplotlib.pyplot as plt

tools = ['uv', 'venv', 'conda']
speed = [10, 3, 5]           # relative speed score
ease = [9, 10, 6]            # relative ease score
non_python = [2, 1, 10]      # non-Python dep support

x = range(len(tools))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - width for i in x], speed, width, label='Speed', color='#3498db', alpha=0.8)
ax.bar(x, ease, width, label='Ease of Use', color='#2ecc71', alpha=0.8)
ax.bar([i + width for i in x], non_python, width, label='Non-Python Deps', color='#e67e22', alpha=0.8)

ax.set_ylabel('Relative Score (1-10)', fontsize=10)
ax.set_title('Environment Tools: Speed, Ease, and Non-Python Support', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(tools)
ax.legend()
ax.set_ylim(0, 11)
ax.grid(True, alpha=0.3, axis='y')

for i, (s, e, n) in enumerate(zip(speed, ease, non_python)):
    ax.text(i - width, s + 0.3, str(s), ha='center', fontweight='bold')
    ax.text(i, e + 0.3, str(e), ha='center', fontweight='bold')
    ax.text(i + width, n + 0.3, str(n), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


> **Rule of thumb:** Start with `uv`. Switch to `conda` only when you need CUDA toolkits or C libraries.


# For This Course: Per-Phase Environment Strategy

---


You could create one environment for the whole course. **Don't.** Different phases need different (sometimes conflicting) dependencies.

```text
ai-engineering-from-scratch/
├── .venv/                    # shared lightweight env for phases 0-3
├── phases/
│   ├── 04-neural-networks/
│   │   └── .venv/            # PyTorch env
│   ├── 05-cnns/
│   │   └── .venv/            # same PyTorch env (symlink or shared)
│   ├── 08-transformers/
│   │   └── .venv/            # might need different transformer versions
│   └── 11-llm-apis/
│       └── .venv/            # API SDKs, no torch needed
```

The script in `code/env_setup.sh` creates the base environment for this course.

**Why per-phase?** Phase 4 (CNNs) might need PyTorch 2.4, while Phase 8 (Transformers) might need Transformers 4.44 which pins an older PyTorch. Separate environments mean neither breaks the other.


# Visual: Per-Phase Environment Layout

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

phases = [
    (0.1, 0.7, 'Phases 0-3\nSetup, Math, ML Fundamentals\n.shared .venv', '#3498db'),
    (0.1, 0.4, 'Phase 4-5\nCNNs, Neural Nets\n.pyproject + torch', '#2ecc71'),
    (0.55, 0.7, 'Phase 8\nTransformers\n.different transformer versions', '#e67e22'),
    (0.55, 0.4, 'Phase 11\nLLM APIs\n.anthropic, openai SDKs', '#9b59b6'),
]

for x, y, label, color in phases:
    rect = patches.Rectangle((x, y-0.18), 0.4, 0.36, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 0.2, y + 0.05, label, ha='center', va='center', color='white', fontweight='bold', fontsize=9)

ax.text(0.3, 0.95, 'Course Repository', ha='center', fontsize=12, fontweight='bold')
ax.text(0.3, 0.05, 'Each phase: independent .venv/, independent dependencies', ha='center', fontsize=10, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Per-Phase Environment Strategy", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **The script:** Run `bash phases/00-setup-and-tooling/06-python-environments/code/env_setup.sh` to create your base course environment.


# pyproject.toml — The Modern Project Config

---


Every Python project should have a `pyproject.toml`. It replaces `setup.py`, `setup.cfg`, and `requirements.txt` in one file.

```toml
[project]
name = "ai-engineering-from-scratch"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "numpy>=1.26",
    "matplotlib>=3.8",
    "jupyter>=1.0",
    "scikit-learn>=1.4",
]

[project.optional-dependencies]
torch = ["torch>=2.3", "torchvision>=0.18"]
llm = ["anthropic>=0.39", "openai>=1.50"]
```

**Then install with dependency groups:**
```bash
uv pip install -e ".[torch]"      # base + PyTorch
uv pip install -e ".[llm]"        # base + LLM SDKs
uv pip install -e ".[torch,llm]"  # everything
```

**Optional dependencies** let you install only what a phase needs, keeping environments lightweight.


# Lockfiles: Guaranteeing Reproducibility

---


A **lockfile** pins every dependency (including transitive ones) to exact versions. Anyone who installs from the lockfile gets exactly the same packages.

```bash
# uv generates uv.lock automatically when using uv add
uv add numpy

# Or compile from pyproject.toml
uv pip compile pyproject.toml -o requirements.lock
uv pip install -r requirements.lock
```

**Commit your lockfile to git.** When someone clones the repo, they install from the lockfile and get identical versions — no "works on my machine" problems.

**Without a lockfile:** `numpy>=1.26` could install 1.26.0 today and 1.26.5 tomorrow, potentially breaking your code.


# Visual: With vs Without Lockfile

---


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Without lockfile
ax = axes[0]
ax.axis('off')
ax.text(0.5, 0.9, 'WITHOUT Lockfile', ha='center', fontsize=12, fontweight='bold', color='#e74c3c')
ax.text(0.1, 0.7, 'pyproject.toml:', fontsize=10, fontweight='bold')
ax.text(0.1, 0.6, 'numpy>=1.26', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.5, 'torch>=2.3', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.4, 'Result:', fontsize=10, fontweight='bold')
ax.text(0.1, 0.3, 'Machine A: numpy 1.26.0, torch 2.3.0', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.2, 'Machine B: numpy 1.26.4, torch 2.4.1', fontsize=9, fontfamily='monospace')
ax.text(0.5, 0.1, 'DIFFERENT RESULTS', ha='center', fontsize=11, fontweight='bold', color='#e74c3c')

# With lockfile
ax = axes[1]
ax.axis('off')
ax.text(0.5, 0.9, 'WITH Lockfile', ha='center', fontsize=12, fontweight='bold', color='#27ae60')
ax.text(0.1, 0.7, 'uv.lock:', fontsize=10, fontweight='bold')
ax.text(0.1, 0.6, 'numpy==1.26.4', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.5, 'torch==2.4.1', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.4, 'Result:', fontsize=10, fontweight='bold')
ax.text(0.1, 0.3, 'Machine A: numpy 1.26.4, torch 2.4.1', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.2, 'Machine B: numpy 1.26.4, torch 2.4.1', fontsize=9, fontfamily='monospace')
ax.text(0.5, 0.1, 'IDENTICAL RESULTS', ha='center', fontsize=11, fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.show()


> **Without a lockfile**, the same `pyproject.toml` can produce different package versions on different machines.
> **With a lockfile**, everyone gets identical versions.


# Common Mistakes and How to Avoid Them

---


### Mistake 1: Installing globally
```bash
pip install torch          # BAD: installs to system Python

source .venv/bin/activate
pip install torch          # GOOD: installs to virtual environment
```

Check where your packages go:
```bash
which python   # should show .venv/bin/python, not /usr/bin/python
which pip      # should show .venv/bin/pip
```

### Mistake 2: Mixing pip and conda
```bash
conda create -n myenv python=3.12
conda activate myenv
conda install pytorch -c pytorch
pip install some-package     # BAD: breaks conda's dependency tracking
conda install some-package   # GOOD: let conda manage everything
```

If you must use pip inside conda (some packages are pip-only), install all conda packages first, then pip packages last.

### Mistake 3: Forgetting to activate
```bash
python train.py                    # uses system Python, missing packages
source .venv/bin/activate
python train.py                    # uses project Python, packages found
```

Your shell prompt should show the environment name:
```
(.venv) $ python train.py
```

### Mistake 4: Committing .venv to git
```bash
echo ".venv/" >> .gitignore
```

Virtual environments are 200MB-2GB. They're local, not portable. Commit `pyproject.toml` and the lockfile instead.


# Visual: The 5 Most Common Mistakes

---


In [ ]:
import matplotlib.pyplot as plt

mistakes = [
    'Global install',
    'pip + conda mixing',
    'Forgot to activate',
    'Committed .venv to git',
    'CUDA version mismatch',
]
frequency = [35, 20, 25, 15, 25]  # relative frequency

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c', '#e67e22', '#f39c12', '#9b59b6', '#34495e']
bars = ax.barh(mistakes, frequency, color=colors, alpha=0.8)
ax.set_xlabel('Relative Frequency (%)', fontsize=10)
ax.set_title('Most Common Environment Mistakes', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, frequency):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontweight='bold')

ax.set_xlim(0, 45)
plt.tight_layout()
plt.show()


> **Global installs and CUDA mismatches** are the most common — check your activation and CUDA versions first.


# CUDA Version Mismatch: The Silent Breaker

---


PyTorch ships CUDA bindings compiled for specific CUDA versions. If the PyTorch CUDA version exceeds your GPU driver's CUDA version, CUDA will not be available.

```bash
nvidia-smi                       # shows driver CUDA version (e.g., 12.4)
python -c "import torch; print(torch.version.cuda)"  # shows PyTorch CUDA version
```

**Rule:** PyTorch CUDA version must be **<=** driver CUDA version.

If they don't match, reinstall PyTorch with a compatible CUDA version:
```bash
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** What problem do virtual environments solve?
> **Explanation:** They isolate project dependencies so different projects can use different package versions. Without them, installing PyTorch 2.4 for one project overwrites PyTorch 2.1 needed by another.

**Q2:** What is a lockfile in Python dependency management?
> **Explanation:** A file that pins every package to an exact version for reproducible installs. It records exact versions of all dependencies, including transitive ones.

**Q3:** How do you verify pip and python are using the virtual environment?
> **Explanation:** Run `which python` and confirm it shows `.venv/bin/python`, not `/usr/bin/python`.

**Q4:** Why is mixing pip and conda problematic?
> **Explanation:** Pip installs bypass conda's dependency solver, so conda no longer knows the true state of the environment. This leads to hard-to-debug conflicts.

**Q5:** PyTorch reports 'CUDA not available' despite having an NVIDIA GPU. Most likely cause?
> **Explanation:** PyTorch was installed with a CUDA version incompatible with your GPU driver. Check with `nvidia-smi` and `torch.version.cuda`.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Run the setup script and verify all checks pass.
```bash
bash phases/00-setup-and-tooling/06-python-environments/code/env_setup.sh
```

**Exercise 2:** Create a second virtual environment, install a different version of numpy in it, and confirm the two environments are isolated.

**Exercise 3:** Write a `pyproject.toml` for a project that needs both PyTorch and the Anthropic SDK.

**Exercise 4:** Deliberately install a package globally (without activating a venv), notice where it goes, then uninstall it.

### Sample pyproject.toml:
```toml
[project]
name = "my-ai-project"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "numpy>=1.26",
]

[project.optional-dependencies]
torch = ["torch>=2.3", "torchvision>=0.18"]
llm = ["anthropic>=0.39"]
```


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Virtual environment | "A venv" | An isolated directory containing a Python interpreter and packages, separate from system Python |
| Lockfile | "Pinned dependencies" | A file listing every package and its exact version, guaranteeing identical installs across machines |
| pyproject.toml | "The new setup.py" | The standard Python project configuration file, replacing setup.py/setup.cfg/requirements.txt |
| Transitive dependency | "A dependency of a dependency" | Package B depends on C; if A depends on B, C is a transitive dependency of A |
| CUDA mismatch | "My GPU isn't working" | PyTorch was compiled for a different CUDA version than your GPU driver supports |
| Dependency hell | "Nothing installs" | Conflicting package version requirements between projects |


# Next Steps

---


When your environments are isolated and working, continue to:
**Lesson 07: Docker for AI**

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
